## Imports & Path Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np


PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "raw"
    / "SPY"
    / "2010"
    / "spy_eod_201001.txt"
)

## Load Data

In [14]:
from src.data.ingestion import load_raw_option_chain

df = load_raw_option_chain(DATA_PATH)

print(df.shape)
df.head()

(18668, 33)


,quote_unixtime,quote_readtime,quote_date,quote_time_hours,underlying_last,expire_date,expire_unix,dte,c_delta,c_gamma,...,p_last,p_delta,p_gamma,p_vega,p_theta,p_rho,p_iv,p_volume,strike_distance,strike_distance_pct
0,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88304,0.00005,...,0.02,-0.00143,0.00010,0.00074,-0.00435,-0.00050,1.36742,0.0,58.3,0.515
1,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88398,0.00005,...,0.00,-0.00160,0.00020,0.00074,-0.00396,-0.00043,1.33476,NaN,57.3,0.506
2,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88657,0.00000,...,0.03,-0.00146,0.00014,0.00106,-0.00369,-0.00004,1.30655,0.0,56.3,0.497
3,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88750,0.00003,...,0.04,-0.00166,0.00017,0.00056,-0.00428,-0.00051,1.27237,0.0,55.3,0.488
4,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88970,0.00007,...,0.04,-0.00167,0.00016,0.00128,-0.00429,-0.00023,1.24421,0.0,54.3,0.479


## Parse Timestamps

In [3]:
# Readable timestamps
df["quote_readtime"] = pd.to_datetime(df["quote_readtime"])
df["quote_date"] = pd.to_datetime(df["quote_date"])
df["expire_date"] = pd.to_datetime(df["expire_date"])

# Unix timestamps
df["quote_from_unix"] = pd.to_datetime(
    df["quote_unixtime"],
    unit="s"
)

df["expire_from_unix"] = pd.to_datetime(
    df["expire_unix"],
    unit="s"
)

## Compare Readable vs Unix Timestamps

In [4]:
df["quote_timestamp_diff"] = (
    df["quote_from_unix"] - df["quote_readtime"]
)

df["quote_timestamp_diff"].value_counts().head(10)

quote_timestamp_diff
0 days 05:00:00    18668
Name: count, dtype: int64

## Check Quote Date Consistency

In [5]:
df["quote_date_from_readtime"] = df["quote_readtime"].dt.normalize()

quote_date_mismatch = (
    df["quote_date_from_readtime"] != df["quote_date"]
)

print("Quote date mismatches:", quote_date_mismatch.sum())

Quote date mismatches: 0


## Check `QUOTE_TIME_HOURS`

In [6]:
df["quote_time_hours_calculated"] = (
    df["quote_readtime"].dt.hour
    + df["quote_readtime"].dt.minute / 60
    + df["quote_readtime"].dt.second / 3600
)

time_difference = (
    df["quote_time_hours"]
    - df["quote_time_hours_calculated"]
)

print("Maximum absolute difference:",
      time_difference.abs().max())

print("Mismatches:",
      (time_difference.abs() > 1e-9).sum())

Maximum absolute difference: 0.0
Mismatches: 0


## Check Expiration Timestamp Consistency

In [7]:
expire_date_from_unix = (
    df["expire_from_unix"].dt.normalize()
)

expiry_date_mismatch = (
    expire_date_from_unix != df["expire_date"]
)

print(
    "Expiration date mismatches:",
    expiry_date_mismatch.sum()
)

Expiration date mismatches: 0


In [8]:
df[
    [
        "expire_date",
        "expire_unix",
        "expire_from_unix"
    ]
].drop_duplicates().head(20)

,expire_date,expire_unix,expire_from_unix
0,2010-01-15,1263589200,2010-01-15 21:00:00
110,2010-02-19,1266613200,2010-02-19 21:00:00
220,2010-03-19,1269028800,2010-03-19 20:00:00
331,2010-03-31,1270065600,2010-03-31 20:00:00
402,2010-06-18,1276891200,2010-06-18 20:00:00
513,2010-06-30,1277928000,2010-06-30 20:00:00
569,2010-09-17,1284753600,2010-09-17 20:00:00
653,2010-09-30,1285876800,2010-09-30 20:00:00
721,2010-12-17,1292619600,2010-12-17 21:00:00
760,2010-12-31,1293829200,2010-12-31 21:00:00


## Inspect Vendor DTE

In [9]:
df["dte"].describe()

count    18668.000000
mean       215.235577
std        210.975353
min          0.000000
25%         63.960000
50%        159.960000
75%        326.000000
max       1082.000000
Name: dte, dtype: float64

In [10]:
df[
    [
        "quote_readtime",
        "quote_unixtime",
        "expire_date",
        "expire_unix",
        "expire_from_unix",
        "dte"
    ]
].head(20)

,quote_readtime,quote_unixtime,expire_date,expire_unix,expire_from_unix,dte
0,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
1,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
2,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
3,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
4,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
5,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
6,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
7,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
8,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0
9,2010-01-04 16:00:00,1262638800,2010-01-15,1263589200,2010-01-15 21:00:00,11.0


## DTE Validation (`QUOTE_DATE` and `EXPIRE_DATE`)

In [11]:
df["dte_from_dates"] = (
    df["expire_date"] - df["quote_date"]
).dt.total_seconds() / (24 * 60 * 60)

dte_diff = df["dte"] - df["dte_from_dates"]

print("Maximum absolute DTE difference:", dte_diff.abs().max())
print("DTE mismatches:", (dte_diff.abs() > 1e-8).sum())

df.loc[
    dte_diff.abs() > 1e-8,
    [
        "quote_readtime",
        "quote_date",
        "expire_date",
        "dte",
        "dte_from_dates",
    ],
].head(20)

Maximum absolute DTE difference: 0.040000000000020464
DTE mismatches: 10381


,quote_readtime,quote_date,expire_date,dte,dte_from_dates
220,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
221,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
222,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
223,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
224,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
225,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
226,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
227,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
228,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0
229,2010-01-04 16:00:00,2010-01-04,2010-03-19,73.96,74.0


## DTE Validation (`QUOTE_UNIXTIME` and `EXPIRE_UNIX`)

In [12]:
df["dte_from_unix"] = (
    df["expire_unix"] - df["quote_unixtime"]
) / (24 * 60 * 60)

df["dte_diff_unix"] = df["dte"] - df["dte_from_unix"]

print(
    "Maximum absolute DTE difference:",
    df["dte_diff_unix"].abs().max()
)

print(
    "DTE mismatches:",
    (df["dte_diff_unix"].abs() > 1e-8).sum()
)

df.loc[
    df["dte_diff_unix"].abs() > 1e-8,
    [
        "quote_readtime",
        "quote_unixtime",
        "expire_date",
        "expire_unix",
        "dte",
        "dte_from_unix",
        "dte_diff_unix",
    ],
].head(20)

Maximum absolute DTE difference: 0.0016666666666651508
DTE mismatches: 10381


,quote_readtime,quote_unixtime,expire_date,expire_unix,dte,dte_from_unix,dte_diff_unix
220,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
221,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
222,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
223,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
224,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
225,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
226,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
227,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
228,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
229,2010-01-04 16:00:00,1262638800,2010-03-19,1269028800,73.96,73.958333,0.001667
